# Data Preparation

## Research Question

*How did COVID-19 affect the spatial and temporal patterns of reported crime in Canberra?*


## Data Sources

#### Crime 
The ACT Crime Statistics $^{[1]}$ dataset was the primary source of data for this project. It contains quarterly crime numbers for each ACT suburb and crime from Q1 2014 to Q2 2025. The original dataset, in .xlsx format, contains seperate worksheets for each ACT region (SA3), each containing a table for each crime with suburbs (SA2s) represented as rows and quarters represented as columns. 

#### Population 
The Regional Population $^{[2]}$ dataset provided yearly population estimates for Australian states, regions, and suburbs between 2001 and 2025. The .xlsx dataset contains worksheets of different statistical area granularities, with rows representing a given area and columns representing area information and the year of population estimate. 

#### Socioeconomic 
Data by Region $^{[3]}$ datasets were used to provide intermittent socioeconomic statistics at the regional (SA3) and suburb (SA2) level for individual statistical areas. These datasets combined data from previous Census records (2011, 2016, and/or 2021), alongside various other sources. Each individual dataset represents a single statistical area with a row representing a measure and columns representing years.


### References
$[1]$: ACT Policing, July 8, 2025, “ACT Crime Statistics,” distributed by Open Data Portal dataACT, Available: https://www.data.act.gov.au/Justice-Safety-and-Emergency/ACT-Crime-Statistics/2egm-dieb/about_data

$[2]$: Australian Bureau of Statistics, March 31, 2025, “Regional population,” distributed by ABS Website, Available: https://www.abs.gov.au/statistics/people/population/regional-population/2024-25#cite-window1

$[3]$: Australian Bureau of Statistics, November 11, 2025, “Data by region,” distributed by ABS Website, Available: https://dbr.abs.gov.au/

## Data Preprocessing
Initial preprocessing was carried out with two main aims:
1. Tidy datasets into long formats
2. Consolidate missing or differing information

Data was obtained from two main sources, the Australian Bureau of Statistics (ABS) and ACT Policing. As such, several inconsistencies were found between the available SA2 information. Primarily, many suburbs were missing from the ACT Policing data, some had different names, and some suburbs were combined to ensure they covered the same geographical area. As not all suburbs in each region could be included, regional (SA3) data was later obtained by grouping suburb data. 

In [28]:
import pandas as pd
from pathlib import Path
import numpy as np

def format_suburb_names(s_name, lower= False, upper = False):
    """
    Formats suburb names from data provided by ABS

    Args:
        s_name: (str) ABS suburb name
        lower: (int) number of words to remove from the start of s_name
        upper: (int) number of words to remove from the end of s_name
    
    Returns:
        (str) Optionally shortened string, with instances of '(ACT)' removed
    """       
    if not lower and not upper:
        s_name = s_name.split()
    else:
        s_name = s_name.split()[lower:upper]
    s_name_filtered = [s for s in s_name if s != '(ACT)']
    return ' '.join(s_name_filtered)


def create_suburb_df(f):
    """
    Reads ABS suburb statistics files in suburb_path to df and concantenates to dataframe 

    Args:
        f: (str) File name in suburb_path
    
    Returns:
        suburb: (pd.Dataframe) Current dataframe including new data from f
    """    
    suburb = pd.read_csv(suburb_path / f)
    suburb['Suburb'] = format_suburb_names(f, lower= 2, upper = -2)
    return suburb


def combine_parkes_ch_suburbs(df, group_list):
    """
    Combines Parkes - North and Parkes - South ABS suburbs to form Parkes & Capital Hill

    Args:
        df: (pd.Dataframe) ABS sourced dataframe
        group_list: (list(str)) Specifies df columns to be used for grouping
    
    Returns:
        df: (pd.Dataframe) Updated dataframe
    """  
    comb_df = df[(df['Suburb'] == 'Parkes - North') | (df['Suburb'] == 'Parkes - South')]
    comb_df = comb_df.groupby(group_list).sum().reset_index()
    parks_ch = comb_df[comb_df['Suburb'] == 'Parkes - NorthParkes - South'].copy()
    parks_ch['Suburb'] = 'Parkes & Capital Hill'
    df = df[(df['Suburb'] != 'Parkes - North') & (df['Suburb'] != 'Parkes - South')]
    df = pd.concat([df, parks_ch], ignore_index=True)
    return df


def group_crimes(df, group_list):
    """
    Groups crime statistics by crime similarity

    Args:
        df: (pd.Dataframe) Crime dataframe with original crime names
        group_list: (list(str)) Specifies df columns to be used for grouping
    
    Returns:
        df: (pd.Dataframe) Grouped crime dataframe
    """

    burglary = df[(df['Crime'] == 'Burglary - Dwellings') | (df['Crime'] == 'Burglary - Other') | 
                        (df['Crime'] == 'Burglary - Shops')]
    burglary = burglary.groupby(group_list).sum().reset_index()
    burglary['Crime'] = 'Burglary'
    df = pd.concat([df, burglary], ignore_index=True)

    robbery = df[(df['Crime'] == 'Robbery - Armed') | (df['Crime'] == 'Robbery - Other')]
    robbery = robbery.groupby(group_list).sum().reset_index()
    robbery['Crime'] = 'Robbery'
    df = pd.concat([df, robbery], ignore_index=True)

    theft = df[(df['Crime'] == 'Theft - Motor Vehicles') | (df['Crime'] == 'Theft - Other')]
    theft = theft.groupby(group_list).sum().reset_index()
    theft['Crime'] = 'Theft'
    df = pd.concat([df, theft], ignore_index=True)

    other = df[(df['Crime'] == 'Other - Against A Person') | (df['Crime'] == 'Other')]
    other = other.groupby(group_list).sum().reset_index()
    other['Crime'] = 'Other Crime'
    df = pd.concat([df, other], ignore_index=True)

    tins = df[(df['Crime'] == 'TINs - Speeding') | (df['Crime'] == 'TINs - Mobile Use') | 
                    (df['Crime'] == 'TINs - Seatbelts') | (df['Crime'] == 'TINs - Other')]
    tins = tins.groupby(group_list).sum().reset_index()
    tins['Crime'] = 'TINs'
    df = pd.concat([df, tins], ignore_index=True)

    # remove crimes which form a group
    df = df[df['Crime'].isin(['Burglary', 'Robbery', 'Theft', 'Other', 
                              'TINs', 'Homicide', 'Family Violence', 'Assault', 
                              'Sexual Assault', 'Property Damage', 'CINs'])]
    return df


def add_regions(df, region_df):
    """
    Creates 'Region' column corresponding to 'Suburb' column

    Args:
        df: (pd.Dataframe) Dataframe containing 'Suburb' column
        region_df: (pd.Dataframe) Dataframe listing suburbs (SA2) and their regions (SA3)
    
    Returns:
        merged_df: (pd.Dataframe) df with added 'Region' column
    """
    merged_df = pd.merge(df, region_df, how='left', on='Suburb')  
    return merged_df


def save_region_suburb(df, group_cols, title, is_crime = False):
    """
    Reorders and saves original dataframe, alongside the same dataframe grouped by 'Region'
    Crime data not saved to final processed path to allow for further manipulation
    Args:
        df: (pd.Dataframe) Dataframe containing 'Suburb' and 'Region' columns
        group_list: (list(str)) Specifies df columns to be used for grouping
        title: (str) Save file name prefix
        is_crime: (binary) Determines file save path
    """

    region_df = df.groupby(group_cols).sum().reset_index().drop(columns=['Suburb'])
    name = title + '_region.csv' 

    col_order = region_df.columns.tolist()
    col_order.insert(1, 'Suburb')
    df = df[col_order]
    name = title + '_suburb.csv'

current_path = Path.cwd()
parent_dir = current_path.parent
raw_path = Path(parent_dir / 'data/raw')
suburb_path = Path(parent_dir / 'data/raw/suburb')

Initially, each dataset containing suburb-level socioeconomic information (Data by Region) was read into a combined dataframe, including the suburb name, and relevant SA2 areas were combined. Data was then filtered to statistics referencing a population number, obtained in all Census years and containing the least amount of missing information.

In [29]:

### PROCESS SUBURB DATA ###

# combine seperate suburb files
raw_f_names = [f.name for f in suburb_path.iterdir() if f.is_file()]
sa2_f_names = [f_name for f_name in raw_f_names if 'Region summary_' in f_name]

suburb_df = create_suburb_df(sa2_f_names[0])
for s in sa2_f_names[1:]:
    suburb_df = pd.concat([suburb_df, create_suburb_df(s)], ignore_index=True)

# combine Parkes and Capitol Hill areas
suburb_df = combine_parkes_ch_suburbs(suburb_df, ['Measure Code', 'Parent Description', 'Description'])

# filter suburb statistics based on data availability
suburb_df['Total Description'] = suburb_df['Parent Description'] + ' : ' + suburb_df['Description']
suburb_df = suburb_df.drop(columns=['Measure Code', 'Parent Description', 'Description', '2015', 
                                    '2017', '2018', '2019', '2020','2022', '2023', '2024','2025'])
suburb_df = suburb_df.dropna()
desc_counts = suburb_df['Total Description'].value_counts()
keep_stats = desc_counts[desc_counts == desc_counts.max()].index
suburb_df = suburb_df[suburb_df['Total Description'].isin(keep_stats)]
suburb_df = suburb_df[suburb_df['Total Description'].str.contains('(no.)')]

print(suburb_df.head())




     2011  2016  2021            Suburb  \
158   3.0   0.0   8.0  ACT - South West   
160   0.0   0.0   0.0  ACT - South West   
162   3.0   0.0   8.0  ACT - South West   
165   0.0   0.0   5.0  ACT - South West   
167   0.0   0.0   0.0  ACT - South West   

                                     Total Description  
158      Labour force status - Census : Employed (no.)  
160    Labour force status - Census : Unemployed (no.)  
162  Labour force status - Census : In the labour f...  
165  Labour force status - Census : Total respondin...  
167  Unpaid assistance to a person with a disabilit...  


C:\Users\molly\AppData\Local\Temp\ipykernel_16272\1129591786.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  suburb_df = suburb_df[suburb_df['Total Description'].str.contains('(no.)')]


The relevant worksheet in the Region Population dataset was processed, only including suburbs in the ACT with complete data from 2014 onwards.

In [30]:
### PROCESS POPULATION DATA ###

pop = pd.read_excel(raw_path / 'population_estimates_aus.xlsx', sheet_name=None)
pop_df = pop['Table 1']
pop_df.columns = pop_df.iloc[3]
pop_df = pop_df.iloc[5:, [1] + list(range(9, len(pop_df.columns)))]
pop_df.columns = ['State', 'Suburb'] + list(pop_df.columns[2:])
pop_df = pop_df[pop_df['State'] == 'Australian Capital Territory']

pop_df['Suburb'] = pop_df['Suburb'].apply(format_suburb_names)
pop_df = combine_parkes_ch_suburbs(pop_df, ['State'])
pop_df = pop_df.drop(columns='State')
# remove suburbs with no/unknown population from start of date range
pop_df = pop_df[pop_df[2014] != 0]
print(pop_df.head())

      Suburb  2001  2002  2003  2004  2005  2006  2007  2008  2009  ...  2016  \
0     Aranda  2565  2544  2525  2517  2527  2522  2549  2551  2553  ...  2447   
1  Belconnen  2867  2884  2929  2957  3111  3163  3379  3538  3809  ...  6743   
2      Bruce  2837  3128  3266  3312  3341  3576  4465  5399  5925  ...  7147   
3  Charnwood  3178  3202  3239  3237  3198  3138  3150  3151  3157  ...  3004   
4       Cook  2976  2980  2974  2959  2927  2917  2949  2972  2985  ...  2902   

   2017  2018  2019  2020  2021  2022  2023  2024  2025  
0  2485  2502  2544  2588  2583  2578  2601  2602  2643  
1  7331  7882  8306  8342  8545  8893  9325  9480  9592  
2  7415  7653  7843  7703  7543  7665  7913  8082  8119  
3  2989  2959  3038  3002  3042  3030  3062  3046  3078  
4  2902  2888  2912  2913  2938  2928  2946  2931  2962  

[5 rows x 26 columns]


Each worksheet from the ACT Crime Statistics dataset was combined into a dataframe, and column names and values formatted appropriately.

In [31]:

### PROCESS CRIME DATA ###

crime_data = pd.read_excel((raw_path / 'Website_Qtrly_Jun25.xlsx'), sheet_name=None)
processed_crime = {}
# format suburb crime data
for k in crime_data.keys():
    suburb_data = crime_data[k].iloc[2:]

    #set temporal column names
    col_names = suburb_data.iloc[1].copy().tolist()
    col_names[0] = 'Suburb'
    suburb_data.columns = col_names

    # remove temporal rows
    suburb_data = suburb_data[suburb_data.iloc[:,0].notna()].reset_index(drop=True)
    # set crime type as column
    suburb_data['Crime'] = np.where(suburb_data.iloc[:,1:].isna().all(axis=1), suburb_data.iloc[:,0], np.nan)
    suburb_data['Crime'] = suburb_data['Crime'].ffill()
    suburb_data = suburb_data[suburb_data.iloc[:,1:].notna().all(axis=1)]
    suburb_data = suburb_data[suburb_data['Suburb'] != 'Total']
    suburb_data['Region'] = k
    processed_crime[k] = suburb_data

# combine data into one df
crime_df = pd.concat(processed_crime.values(), ignore_index=True)
crime_df.columns = np.where((crime_df.columns != 'Suburb') | (crime_df.columns != 'Crime'), 
                            crime_df.columns.str.replace(r'\s+\S+$', '', regex=True), crime_df.columns)

# format crime names
crime_df['Crime'] = crime_df['Crime'].str.title()

crime_df['Crime'] = crime_df['Crime'].replace({'Assault - Fv': 'Family Violence', 'Assault - Non-Fv': 'Assault', 
                                                'Other Offences Against A Person': 'Other - Against A Person',
                                                'Burglary Dwellings': 'Burglary - Dwellings',
                                                'Burglary Shops': 'Burglary - Shops', 
                                                'Burglary Other': 'Burglary - Other',
                                                'Motor Vehicle Theft' : 'Theft - Motor Vehicles',
                                                'Theft (Excluding Motor Vehicles)': 'Theft - Other',
                                                'Other Offences': 'Other', 'Tins Speeding': 'TINs - Speeding',
                                                'Tins Mobile Use': 'TINs - Mobile Use', 'Tins Seatbelts': 'TINs - Seatbelts',
                                                'Tins Other': 'TINs - Other', 'Cins': 'CINs'})

# format crime suburb names
crime_df['Suburb'] = crime_df['Suburb'].str.title()
crime_df['Suburb'] = np.where(crime_df['Suburb'] == 'Mckellar', 'McKellar', crime_df['Suburb'])
crime_df['Suburb'] = np.where(crime_df['Suburb'] == 'City', 'Civic', crime_df['Suburb'])
# combine Parkes and Capitol Hill areas
crime_pch = crime_df[(crime_df['Suburb'] == 'Parkes') | (crime_df['Suburb'] == 'Capital Hill')]
crime_pch = crime_pch.groupby('Crime').sum().reset_index()
# warning: crossover between inner north and south 
crime_pch['Suburb'] = 'Parkes & Capital Hill'
crime_pch['Region'] = 'Inner South'
crime_df = pd.concat([crime_df, crime_pch], ignore_index=True)
crime_df = crime_df[(crime_df['Suburb'] != 'Parkes') & (crime_df['Suburb'] != 'Capital Hill')]
# remove Hume 
crime_df = crime_df[crime_df['Region'] != 'Other']
print(crime_df.head())

      Suburb 2014 Q1 2014 Q2 2014 Q3 2014 Q4 2015 Q1 2015 Q2 2015 Q3 2015 Q4  \
0     Aranda       0       0       0       0       0       0       0       0   
1  Belconnen       0       0       0       0       0       0       0       0   
2      Bruce       0       0       0       0       0       2       0       0   
3  Charnwood       0       0       0       0       0       0       0       0   
4       Cook       0       0       0       0       0       0       0       0   

  2016 Q1  ... 2023 Q3 2023 Q4 2024 Q1 2024 Q2 2024 Q3 2024 Q4 2025 Q1  \
0       0  ...       0       0       0       0       0       0       0   
1       0  ...       0       0       0       0       0       0       0   
2       0  ...       0       1       0       0       0       0       0   
3       0  ...       0       0       0       0       0       0       0   
4       0  ...       0       0       0       0       0       0       0   

  2025 Q2     Crime     Region  
0       0  Homicide  Belconnen  
1       

All three dataframes were filtered to only include information from suburbs in the intersection of the three datasets; a total of 103 suburbs. 
They were then reshaped into long format, and corresponding region names were added to the ABS-sourced data.

In [32]:

# keep only data for suburbs found in intersection of suburb and crime dataframes
remove_s_suburb = list(set(suburb_df['Suburb'].unique()) - set(pop_df['Suburb'].unique()))
suburb_df = suburb_df[~suburb_df['Suburb'].isin(remove_s_suburb)]
remove_s_crime = list(set(crime_df['Suburb'].unique()) - set(suburb_df['Suburb'].unique()))
crime_df = crime_df[~crime_df['Suburb'].isin(remove_s_crime)]
remove_s_pop = list(set(pop_df['Suburb'].unique()) - set(crime_df['Suburb'].unique()))
pop_df = pop_df[~pop_df['Suburb'].isin(remove_s_pop)]
remove_s_suburb = list(set(suburb_df['Suburb'].unique()) - set(pop_df['Suburb'].unique()))
suburb_df = suburb_df[~suburb_df['Suburb'].isin(remove_s_suburb)]
print(crime_df['Suburb'].nunique())
print(pop_df['Suburb'].nunique())
print(suburb_df['Suburb'].nunique())

103
103
103


In [ ]:


# reshape into long format
crime_long = crime_df.melt( id_vars=['Suburb', 'Crime', 'Region'], var_name='Year_Quarter', value_name='Number')
crime_long[['Year', 'Quarter']] = crime_long['Year_Quarter'].str.split(' ', expand=True)
crime_long = crime_long.drop(columns=['Year_Quarter'])
crime_long['Year'] = crime_long['Year'].astype(int)
pop_long = pop_df.melt( id_vars='Suburb', var_name='Year', value_name='Population')
pop_long = pop_long[pop_long['Year'].isin(crime_long['Year'])]
suburb_long = suburb_df.melt( id_vars=['Suburb', 'Total Description'], var_name='Year', value_name='Value')

# add region data where missing
region_suburbs = crime_long[['Region', 'Suburb']].drop_duplicates(['Region', 'Suburb'])
pop_long = add_regions(pop_long, region_suburbs)
suburb_long = add_regions(suburb_long, region_suburbs)

print(crime_long.head())
print(pop_long.head())
print(suburb_long.head())


      Suburb     Crime     Region Number  Year Quarter
0     Aranda  Homicide  Belconnen      0  2014      Q1
1  Belconnen  Homicide  Belconnen      0  2014      Q1
2      Bruce  Homicide  Belconnen      0  2014      Q1
3  Charnwood  Homicide  Belconnen      0  2014      Q1
4       Cook  Homicide  Belconnen      0  2014      Q1
      Suburb  Year Population     Region
0     Aranda  2014       2477  Belconnen
1  Belconnen  2014       6332  Belconnen
2      Bruce  2014       7010  Belconnen
3  Charnwood  2014       3074  Belconnen
4       Cook  2014       2949  Belconnen
  Suburb                                  Total Description  Year  Value  \
0  Acton  Language - Census : Aboriginal and Torres Stra...  2011    0.0   
1  Acton     Language - Census : Speaks English at home (%)  2011   66.7   
2  Acton  Engagement in employment, education or trainin...  2011  100.0   
3  Acton  Engagement in employment, education or trainin...  2011    0.0   
4  Acton  Engagement in employment, educatio

In [ ]:

# filter suburb statistics based on data availability



Index(['Closing the Gap Target 9A - Census : Living in a dwelling requiring three additional bedrooms (%)',
       'Closing the Gap Target 9A - Census : Living in a dwelling requiring three additional bedrooms (no.)',
       'Closing the Gap Target 9A - Census : Living in a dwelling requiring two additional bedrooms (no.)',
       'Closing the Gap Target 9A - Census : Living in a dwelling requiring two additional bedrooms (%)',
       'Closing the Gap Target 9A - Census : Living in a dwelling requiring one additional bedroom (no.)',
       'Closing the Gap Target 9A - Census : Living in a dwelling requiring one additional bedroom (%)',
       'Closing the Gap Target 6 - People aged 25-34 years - Census : Completed tertiary qualification of Certificate III or above (%)',
       'Closing the Gap Target 6 - People aged 25-34 years - Census : Total applicable population aged 25-34 years (no.)',
       'Tenure type - Census : Renter (no.)',
       'Tenure type - Census : Renter (%)',
      